# Data Description

In [ ]:
# 12 Data Layer Lengkap untuk EY Water Quality Challenge 2026

| No | Dataset                  | Tipe Data              | Static / Time-dependent | Sumber                  | Cara Didapat              | Nama File di Stage                          | Fungsi di data_ingestion.py                     | Peran Utama dalam Prediksi Water Quality                          | Gunanya untuk Alkalinity / EC / DRP                          | Butuh Cloud Cleaning? | Boost Potensial     |
|----|--------------------------|------------------------|--------------------------|-------------------------|---------------------------|---------------------------------------------|--------------------------------------------------|-------------------------------------------------------------------|-------------------------------------------------------------|-----------------------|---------------------|
| 1  | Water Quality (Target)   | CSV (Target)           | Time-dependent          | EY                      | Sudah disediakan          | `water_quality_training_dataset.csv`        | Tidak perlu (sudah di-load)                      | Variabel target (Alkalinity, EC, DRP)                             | -                                                           | -                     | -                   |
| 2  | Landsat Features         | CSV (Pre-extracted)    | Time-dependent          | EY + Planetary          | Sudah diekstrak           | `landsat_features_training.csv`             | Tidak perlu                                      | Vegetasi, air, bare soil, mining signature                        | Deteksi runoff & erosi                                      | Sudah dibersihkan     | Tinggi              |
| 3  | TerraClimate             | CSV (Pre-extracted)    | Time-dependent          | EY + Planetary          | Sudah diekstrak           | `terraclimate_features_training.csv`        | Tidak perlu                                      | Evapotranspirasi, defisit tekanan uap                             | Pengaruh kekeringan & konsentrasi ion                       | -                     | Sedang              |
| 4  | SoilGrids 2.0            | API Raster             | **Static**              | ISRIC                   | API langsung              | Tidak ada file                              | `fetch_soilgrids_properties()`                   | pH, clay, sand, organic carbon, CEC                               | **Alkalinity** (pH + carbonate), **EC** (salinity), **DRP** (adsorpsi clay) | Tidak                 | **Sangat Tinggi**   |
| 5  | Elevation (DEM)          | API Point              | **Static**              | OpenTopoData            | API langsung              | Tidak ada file                              | `fetch_elevation()`                              | Ketinggian & slope                                                | Arah aliran & runoff speed                                  | Tidak                 | Sedang              |
| 6  | Weather 7-day lag        | API Time-series        | Time-dependent          | Open-Meteo              | API langsung              | Tidak ada file                              | `fetch_historical_weather()`                     | Curah hujan & angin 7 hari sebelum sampling                       | **DRP** & **EC** naik saat hujan deras (runoff)             | Tidak                 | Tinggi              |
| 7  | OSM Pollution            | API Vector             | **Static**              | Overpass                | API langsung              | Tidak ada file                              | `fetch_osm_pollution_counts()`                   | Jumlah tambang, wastewater plant, farmland, jalan dalam radius 1 km | **DRP** (pertanian), **EC/Alkalinity** (mining)             | Tidak                 | Tinggi              |
| 8  | WorldPop                 | Raster (100m)          | **Static**              | WorldPop                | Link download             | `zaf_pop_2025_CN_100m_R2025A_v1.tif`        | `fetch_static_raster_value("worldpop")`          | Kepadatan penduduk 100 m                                          | Proxy limbah domestik → **DRP** & **EC**                    | Tidak                 | Sedang              |
| 9  | BasinATLAS               | Vector GDB             | **Static**              | HydroSHEDS              | Link download             | `BasinATLAS_v1.gdb`                         | `fetch_hydroatlas_attributes()`                  | Luas catchment hulu, populasi basin, % agrikultur, slope          | Akumulasi polutan dari hulu sungai                          | Tidak                 | Tinggi              |
| 10 | RiverATLAS               | Vector GDB             | **Static**              | HydroSHEDS              | Link download             | `RiverATLAS_v1.gdb`                         | `fetch_riveratlas_attributes()`                  | Discharge rata-rata, river order, lebar sungai                    | Pengenceran & transport polutan                             | Tidak                 | Tinggi              |
| 11 | SANLC 2022 (Albers)      | Raster + VAT (20m)     | **Static**              | DFFE                    | Manual download           | `SA_NLC_2022_ALBERS.tif` + `.vat.dbf`       | `fetch_static_raster_value("sanlc2022")`         | 73 kelas lahan spesifik SA (mining, cropland, urban, dll)         | **DRP** (pertanian), **EC/Alkalinity** (mining & urban)     | Tidak                 | **Sangat Tinggi**   |
| 12 | SANLC 2020 (Albers)      | Raster + VAT (20m)     | **Static**              | DFFE                    | Manual download           | `SA_NLC_2020_ALBERS.tif` + `.vat.dbf`       | `fetch_static_raster_value("sanlc2020")`         | Versi 2020 (untuk perbandingan perubahan)                         | Deteksi perubahan lahan antar tahun                         | Tidak                 | Sedang-Tinggi       |

**Keterangan:**
- **Static** = Tidak berhubungan dengan tanggal sampling (hanya dihitung sekali)
- **Time-dependent** = Berubah sesuai tanggal sampling (harus diambil per baris)
- **Prioritas Boost** diurutkan dari yang paling berpengaruh untuk parameter Alkalinity, EC, dan Dissolved Reactive Phosphorus

---

# Setup & Install

In [ ]:
import time

start_time = time.time()
print("Installing a special library for Notebook 01...")

!pip install uv
!uv pip install geopandas pyarrow tqdm requests

elapsed_time = time.time() - start_time
print(f"\n✓ Instalasi selesai! (Waktu: {elapsed_time:.1f} detik)")

In [ ]:
%%sql -r dataframe_1
ALTER NOTEBOOK "USER$"."PUBLIC"."WATER_QUALITY_PREDICTION" 
SET EXTERNAL_ACCESS_INTEGRATIONS = (EY_DATA_ACCESS_INTEGRATION);

In [ ]:
import requests
import geopandas as gpd
import os
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE SNOWFLAKE_LEARNING_DB").collect()
session.sql("USE SCHEMA PUBLIC").collect()

data_stage_path = "@SNOWFLAKE_LEARNING_DB.PUBLIC.EXTERNAL_DATA_STAGE"
print("Snowflake session and Module ready!")

# download data

In [ ]:
data_to_download = {
    "zaf_pop_2025_CN_100m_R2025A_v1.tif": "https://data.worldpop.org/GIS/Population/Global_2015_2030/R2025A/2025/ZAF/v1/100m/constrained/zaf_pop_2025_CN_100m_R2025A_v1.tif",
    "BasinATLAS_v1.gdb.zip": "https://figshare.com/ndownloader/files/20082137",
    "RiverATLAS_v1.gdb.zip": "https://figshare.com/ndownloader/files/20087321"
}

for file_name, url in data_to_download.items():
    local_path = f"/tmp/{file_name}"

    if os.path.exists(local_path):
        print(f"Finding file remains {file_name} previously. Delete to re-download cleanly....")
        os.remove(local_path)
        
    print(f" Initiating a connection to {file_name} ...")
    response = requests.get(url, stream=True, timeout=900)
    response.raise_for_status()
    
    total_size_in_bytes = int(response.headers.get('content-length', 0))
    block_size = 1024 * 1024 # 1 Megabyte chunk
    

    progress_bar = tqdm(total=total_size_in_bytes, unit='iB', unit_scale=True, desc=file_name)
    
    with open(local_path, 'wb') as f:
        for chunk in response.iter_content(chunk_size=block_size): 
            if chunk: 
                f.write(chunk)
                progress_bar.update(len(chunk)) 
                
    progress_bar.close()
    

    if total_size_in_bytes != 0 and progress_bar.n != total_size_in_bytes:
        print(f"ERROR: An error occurred, {file_name} did not download completely.")
    else:
        print(f" {file_name} successfully downloaded in full!\n")

# Convert to parquet

In [ ]:
print("Begin Convertion GeoDatabase (.gdb) to GeoParquet (.parquet)...")

# Konversi BasinATLAS
if os.path.exists("/tmp/BasinATLAS_v1.gdb.zip"):
    print("Reading BasinATLAS GDB to RAM...")
    basin_gdf = gpd.read_file("zip:///tmp/BasinATLAS_v1.gdb.zip")
    print("Converting BasinATLAS to Parquet...")
    basin_gdf.to_parquet("/tmp/BasinATLAS_v1.parquet", compression='snappy')
    print("✓ BasinATLAS Parquet complete!")
else:
    print("File ZIP BasinATLAS tidak ditemukan!")

# Konversi RiverATLAS
if os.path.exists("/tmp/RiverATLAS_v1.gdb.zip"):
    print("\nReading RiverATLAS GDB to RAM...")
    river_gdf = gpd.read_file("zip:///tmp/RiverATLAS_v1.gdb.zip")
    print("Converting RiverATLAS to Parquet...")
    river_gdf.to_parquet("/tmp/RiverATLAS_v1.parquet", compression='snappy')
    print("✓ RiverATLAS Parquet complete!")
else:
    print("File ZIP RiverATLAS tidak ditemukan!")

# Upload to stage

In [ ]:
print("Uploading file statis final ke Snowflake Stage...")

files_to_upload = [
    "/tmp/zaf_pop_2025_CN_100m_R2025A_v1.tif",
    "/tmp/BasinATLAS_v1.parquet", 
    "/tmp/RiverATLAS_v1.parquet"  
]

for file_path in files_to_upload:
    if os.path.exists(file_path):
        file_name = os.path.basename(file_path)
        print(f"Mengunggah {file_name}...")
        session.file.put(f"file://{file_path}", data_stage_path, auto_compress=False, overwrite=True)
    else:
        print(f"Skip {file_path} file was not found at /tmp/")

print("\n✓ Automation process Complete! Mari kita cek isi Stage saat ini:")
session.sql(f"LIST {data_stage_path}").show()